In [96]:
import pandas as pd
from datetime import datetime, timedelta

### 计算工作时长的函数

In [97]:
def is_workday(date, holidays):
    # 判断是否为工作日（非周末且非节假日）
    return date not in holidays

def calculate_work_duration(start_time, end_time, holidays):
    # 计算工作时间（排除节假日和周末）
    current = start_time
    work_duration = timedelta()
    
    while current < end_time:
        next_day = (current + timedelta(days=1)).replace(hour=0, minute=0, second=0)
        if is_workday(current.date(), holidays):
            work_duration += min(next_day, end_time) - current
        current = next_day
    
    return work_duration.total_seconds() / (24 * 3600)  # 转换为天数

### 读取所需数据文件

In [98]:
base_df = pd.read_excel(r"C:\Users\zhangbon\Desktop\8月-流程总时长分析统计-V2.xlsx", engine='openpyxl')
holiday_df = pd.read_excel(r"E:\000000我的事项\2025-08\1131统计\2025非工作日清单-截至0901.xlsx", engine='openpyxl')
holidays = [datetime.strptime(str(date).strip(), '%Y-%m-%d %H:%M:%S').date() for date in holiday_df['方太假期']]

### 流程数据处理

In [99]:

base_df['流程提交时间'] = base_df['流程提交时间'].astype(str).str.replace('/','-')
base_df['流程结束时间'] = base_df['流程结束时间'].astype(str).str.replace('/','-')
base_df['流程提交时间'] = pd.to_datetime(base_df['流程提交时间'], format='%Y-%m-%d %H:%M:%S')
base_df['流程结束时间'] = pd.to_datetime(base_df['流程结束时间'], format='%Y-%m-%d %H:%M:%S')
base_df['流程总时长（工作日）'] = base_df.apply(
    lambda row: calculate_work_duration(row['流程提交时间'],row['流程结束时间'],holidays),
    axis=1
)
base_df['流程节点总数'] = base_df.groupby(['所属IT系统','流程名称'])['流程编号（每次提起后算1个编号）'].transform('count')
base_df['流程总数'] = base_df.groupby(['所属IT系统','流程名称'])['流程编号（每次提起后算1个编号）'].transform('nunique')
base_df['平均节点数'] = base_df['流程节点总数']/base_df['流程总数']



In [100]:
# base_df

### 统计分析

In [101]:

cal_df = pd.DataFrame()
#此时只保留了流程维度，节点维度已经被去重了
process_df = base_df[['所属IT系统','流程名称','流程编号（每次提起后算1个编号）','流程总时长（工作日）','平均节点数']].drop_duplicates()
# 计算频次，平均总时长，总时长的50分位数
process_df['频次'] = process_df.groupby(['所属IT系统','流程名称'])['流程编号（每次提起后算1个编号）'].transform('count')
process_df['平均总时长'] = process_df.groupby(['所属IT系统','流程名称'])['流程总时长（工作日）'].transform('mean')
process_df['50分值-总时长'] = process_df.groupby(['所属IT系统','流程名称'])['流程总时长（工作日）'].transform('quantile',0.5)
process_df['90分值-总时长'] = process_df.groupby(['所属IT系统','流程名称'])['流程总时长（工作日）'].transform('quantile',0.9)
process_df['95分值-总时长'] = process_df.groupby(['所属IT系统','流程名称'])['流程总时长（工作日）'].transform('quantile',0.95)
process_df

,所属IT系统,流程名称,流程编号（每次提起后算1个编号）,流程总时长（工作日）,平均节点数,频次,平均总时长,50分值-总时长,90分值-总时长,95分值-总时长
0,EHR,退休申请审批流程,2025042310000825774,81.314028,4.058824,17,14.722407,8.372801,30.938778,41.362889
3,EHR,退休申请审批流程,2025061910000849225,30.647894,4.058824,17,14.722407,8.372801,30.938778,41.362889
8,EHR,异动流程,2025072110000867843,4.700799,7.235294,136,3.367436,2.708924,7.916892,11.217758
10,EHR,补签审批流程,2025081110000881153,1.535093,1.093971,5374,0.355505,0.016528,1.011294,1.907198
11,EHR,请假审批流程,2025081110000881157,0.479514,1.336246,3548,0.456327,0.045874,1.280160,2.025363
...,...,...,...,...,...,...,...,...,...,...
124296,EPS,RFX核价审批-非采,622376XJ20250800093,0.317245,3.495238,210,0.734091,0.619641,1.371015,1.776255
124299,EPS,项目验收审批,622751YS202508000304,0.018981,1.919753,162,1.864420,0.769103,3.373587,7.097991
124301,EPS,采购合同审批,623200N250818015,1.776644,2.976812,345,0.953010,0.530058,2.726458,2.826667
124305,EPS,采购订单审批(零星类)-生产,624747PO25082200000134,0.003345,1.000000,629,0.075951,0.025093,0.194509,0.300752


### 统计分析结果的整理和导出

In [103]:
out_df = process_df[['所属IT系统','流程名称','频次','平均节点数','平均总时长','50分值-总时长','90分值-总时长','95分值-总时长']].drop_duplicates()
out_df = out_df.reset_index(drop=True)
out_df.to_excel('C:\\Users\\zhangbon\\Desktop\\流程分析-0904-2.xlsx',index=False)
